In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Landing Timestamp")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:
bronze_df = spark.read.table(f"adb_healthcare.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")


In [0]:
from pyspark.sql import functions as F

target_table = f"adb_healthcare.silver.{table_name}"
normalized_load_type = (load_type or "FULL").strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

if normalized_load_type == "MERGE":
    if not merge_key_list:
        raise ValueError("merge_keys must be provided when load_type is MERGE.")

    missing_merge_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_merge_keys}")

updated_df = (
    bronze_df
        .withColumn("insert_timestamp", F.current_timestamp())
        .withColumn("update_timestamp", F.current_timestamp())
)


In [0]:
if normalized_load_type == "FULL":
    updated_df.write.mode("overwrite").saveAsTable(target_table)

elif normalized_load_type == "APPEND":
    updated_df.write.mode("append").saveAsTable(target_table)

elif normalized_load_type == "MERGE":
    if not spark.catalog.tableExists(target_table):
        updated_df.write.mode("overwrite").saveAsTable(target_table)
    
    else:
        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)

        merge_condition = " AND ".join(
            [f"target.{key} = source.{key}" for key in merge_key_list]
        )

        update_assignments = ", ".join(
            [
                f"target.{column} = source.{column}" for column in updated_df.columns if column != "insert_timestamp"
            ]
        )

        insert_columns = ", ".join([f"{column}" for column in updated_df.columns])
        insert_values = ", ".join([f"source.{column}" for column in updated_df.columns])

        sql_script = f"""
            MERGE INTO {target_table} as target
            USING {source_view} as source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})
        """

        display(sql_script)

        spark.sql(sql_script)

else:
    raise ValueError("load_type must be one of: FULL, APPEND, MERGE")


In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))

In [0]:
%skip
%sql
select * from adb_healthcare.silver.hospitals

In [0]:
%skip
display(updated_df)